In [0]:
%run ./01_config

In [0]:
"""
14_alternative_policies.py  —  Alternative survival and optimisation approaches

Part of the SAP PM cost-aware maintenance optimisation pipeline.
Run order is filename order: 01 through 15.
"""

# 14 — Alternative survival and optimisation approaches

# The 1.8% interval-optimisation result against a 15% criterion is reported in Chapter 5 but
# not decomposed. This notebook separates the possible explanations, which have different
# implications for the contribution:

# 1. A ceiling — the incumbent policy is already close to optimal, so little is available
#    to any method. Measured as the saving at the true optimum.
# 2. Estimation error — the survival models recover parameters too poorly to locate the
#    optimum. Measured as the gap between the true-optimum and fitted-optimum savings.
# 3. Policy-family misspecification — age replacement is the wrong policy class.
#    Measured by comparing against block replacement and a non-parametric alternative.
# 4. Method choice — a different estimator would do better. Measured by re-deriving
#    intervals from each rung of the model ladder.

# Every policy is scored the same way: its recommended interval is evaluated under the
# disclosed true parameters, so what is compared is realised cost, not predicted cost.

# Shared configuration from '01_config' is assumed to be in scope.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

use_project_schema()

INK, MUTED, GRID = "#1c1c1c", "#8a8a8a", "#e0e0e0"
ACCENT, WARM, GREEN, PURPLE = "#2b6cb0", "#c05621", "#2f855a", "#6b46c1"
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 300, "font.size": 9,
    "axes.edgecolor": MUTED, "axes.labelcolor": INK, "text.color": INK,
    "xtick.color": MUTED, "ytick.color": MUTED, "axes.grid": True,
    "grid.color": GRID, "grid.linewidth": 0.6, "axes.axisbelow": True,
    "axes.spines.top": False, "axes.spines.right": False,
    "figure.facecolor": "white", "savefig.facecolor": "white",
})
FIG = f"{FIGURES}/{DATASET_VERSION}_"

def emit(fig, name):
    fig.tight_layout()
    fig.savefig(f"{FIG}{name}.png", bbox_inches="tight")
    fig.savefig(f"{FIG}{name}.svg", bbox_inches="tight", format="svg")
    print(f"saved {FIG}{name}.png / .svg")
    plt.show(); plt.close(fig)

# Inputs

rec = spark.table(tbl("results_parameter_recovery")).toPandas().set_index("equipment_class")
costs = spark.table(tbl("gold_class_cost_params")).toPandas().set_index("equipment_class")
cycles = spark.table(tbl("gold_incumbent_cycles")).toPandas().set_index("equipment_class")
gt = spark.table(tbl("bronze_ground_truth_params")).toPandas().set_index("EQTYP")
for c in ["weibull_beta", "weibull_eta_days"]:
    gt[c] = gt[c].astype(float)
iv = spark.table(tbl("gold_survival_intervals")).toPandas()
downtime = spark.sql(f"""
    SELECT e.equipment_class, AVG(o.downtime_valuation) AS d
    FROM {tbl('silver_order')} o JOIN {tbl('silver_equipment')} e
      ON o.equipment_id = e.equipment_id
    WHERE o.cost_type = 'BREAKDOWN' GROUP BY e.equipment_class
""").toPandas().set_index("equipment_class")["d"].fillna(0.0)
fleet_n = (spark.table(tbl("silver_equipment")).groupBy("equipment_class").count()
           .toPandas().set_index("equipment_class")["count"])
print(f"{len(rec)} classes")

# Cost models for two policy families

# Age replacement renews preventively at age T or correctively at failure, whichever
# comes first — the policy used throughout the study. Block replacement renews at fixed
# calendar intervals regardless of intervening failures, which is closer to how a calendar
# maintenance plan behaves in SAP, and is therefore the fairer comparator for the incumbent.
# Block replacement is costed through the renewal function, approximated numerically.

def surv(t, b, eta):
    return np.exp(-(np.asarray(t, float) / eta) ** b)

def cost_rate_age(T, b, eta, cp, cf):
    grid = np.linspace(0.0, T, 400)
    ST = float(surv(T, b, eta))
    return (cp * ST + cf * (1 - ST)) / max(float(np.trapezoid(surv(grid, b, eta), grid)), 1e-9)

def renewal_function(T, b, eta, steps=400):
    """Expected number of failures in [0, T] for a renewal process, by discrete convolution."""
    dt = T / steps
    t = np.linspace(dt, T, steps)
    F = 1 - surv(t, b, eta)
    m = np.zeros(steps)
    f = np.diff(np.r_[0.0, F])
    for i in range(steps):
        m[i] = F[i] + np.sum(f[:i] * m[i - 1::-1][:i]) if i else F[i]
    return float(m[-1])

def cost_rate_block(T, b, eta, cp, cf):
    return (cp + cf * renewal_function(T, b, eta)) / T

def optimise(fn, b, eta, cp, cf, hi_mult=3.0):
    Ts = np.linspace(20.0, eta * hi_mult, 220)
    r = np.array([fn(t, b, eta, cp, cf) for t in Ts])
    i = int(r.argmin())
    lo, hi = Ts[max(i - 1, 0)], Ts[min(i + 1, len(Ts) - 1)]
    Ts2 = np.linspace(lo, hi, 40)
    r2 = np.array([fn(t, b, eta, cp, cf) for t in Ts2])
    j = int(r2.argmin())
    return float(Ts2[j]), float(r2[j])

# Non-parametric and heuristic alternatives

# Two comparators that require no distributional assumption at all. The **Kaplan–Meier
# optimum** substitutes the empirical survival curve into the age-replacement cost rate,
# which is what a practitioner could do without fitting anything. The MTBF heuristic
# sets the interval to a fraction of observed mean time between failures — a rule of thumb
# in common industrial use, and the honest baseline the framework must beat to be worth
# adopting.

def km_curve(dur, evt):
    d = pd.DataFrame({"t": np.asarray(dur, float), "e": np.asarray(evt)}).sort_values("t")
    n, s = len(d), 1.0
    ts, ss = [0.0], [1.0]
    for t, grp in d.groupby("t"):
        k = grp.e.sum()
        if k:
            s *= (1 - k / n)
        ts.append(float(t)); ss.append(s); n -= len(grp)
    return np.array(ts), np.array(ss)

def km_optimum(ts, ss, cp, cf):
    best, bestT = np.inf, None
    for i in range(2, len(ts)):
        T, ST = ts[i], ss[i]
        area = float(np.trapezoid(ss[:i + 1], ts[:i + 1]))
        if area <= 0:
            continue
        r = (cp * ST + cf * (1 - ST)) / area
        if r < best:
            best, bestT = r, T
    return bestT, best

# Evaluate every policy under the true parameters

rows = []
for cls in sorted(rec.index):
    b_t, e_t = float(gt.loc[cls, "weibull_beta"]), float(gt.loc[cls, "weibull_eta_days"])
    b_f, e_f = float(rec.loc[cls, "beta_fitted"]), float(rec.loc[cls, "eta_fitted"])
    cp = float(costs.loc[cls, "preventive_cost_median"])
    cf = float(costs.loc[cls, "breakdown_cost_median"]) + float(downtime.get(cls, 0.0))
    as_is = float(cycles.loc[cls, "cycle_days_mean"])
    base_rate = cost_rate_age(as_is, b_t, e_t, cp, cf)

    g = iv[iv.equipment_class == cls]
    ts, ss = km_curve(g.duration_days.values, g.event_observed.values)
    mtbf = float(g.loc[g.event_observed == 1, "duration_days"].mean()) if g.event_observed.sum() else as_is

    cand = {}
    cand["incumbent calendar"] = as_is
    cand["age replacement (fitted)"] = optimise(cost_rate_age, b_f, e_f, cp, cf)[0]
    cand["age replacement (true params)"] = optimise(cost_rate_age, b_t, e_t, cp, cf)[0]
    cand["block replacement (fitted)"] = optimise(cost_rate_block, b_f, e_f, cp, cf)[0]
    km_T, _ = km_optimum(ts, ss, cp, cf)
    cand["Kaplan-Meier optimum"] = km_T if km_T else as_is
    cand["MTBF heuristic (0.7 x MTBF)"] = 0.7 * mtbf

    for policy, T in cand.items():
        realised = cost_rate_age(float(T), b_t, e_t, cp, cf)
        rows.append({"equipment_class": cls, "policy": policy,
                     "interval_days": int(round(float(T))),
                     "realised_saving_pct": round(100 * (base_rate - realised) / base_rate, 2),
                     "weight": float(fleet_n.get(cls, 1)) * base_rate})

alt = pd.DataFrame(rows)
fleet = (alt.groupby("policy")
            .apply(lambda d: np.average(d.realised_saving_pct, weights=d.weight))
            .rename("fleet_saving_pct").reset_index()
            .sort_values("fleet_saving_pct", ascending=False))
fleet["fleet_saving_pct"] = fleet.fleet_saving_pct.round(2)
display(spark.createDataFrame(fleet))

# Confidence-gated recommendations

# The optimizer currently issues a recommendation for every class, including those where
# parameter recovery failed — and Chapter 5 shows those are exactly where it does harm. A
# gate that withholds the recommendation where the evidence is thin is tested here.

# This rule was specified after observing which classes performed badly. It is therefore
# not a pre-registered result and is reported as a diagnosed post-hoc improvement, not as a
# criterion outcome.

MIN_FAILURES, MAX_CENSORING = 40, 0.85
gate_rows = []
for cls in sorted(rec.index):
    fails = int(rec.loc[cls, "train_failures"])
    cens = float(rec.loc[cls, "censoring_rate"])
    gated = (fails < MIN_FAILURES) or (cens > MAX_CENSORING)
    b_t, e_t = float(gt.loc[cls, "weibull_beta"]), float(gt.loc[cls, "weibull_eta_days"])
    b_f, e_f = float(rec.loc[cls, "beta_fitted"]), float(rec.loc[cls, "eta_fitted"])
    cp = float(costs.loc[cls, "preventive_cost_median"])
    cf = float(costs.loc[cls, "breakdown_cost_median"]) + float(downtime.get(cls, 0.0))
    as_is = float(cycles.loc[cls, "cycle_days_mean"])
    base_rate = cost_rate_age(as_is, b_t, e_t, cp, cf)
    T = as_is if gated else optimise(cost_rate_age, b_f, e_f, cp, cf)[0]
    gate_rows.append({"equipment_class": cls, "train_failures": fails,
                      "censoring_rate": round(cens, 3), "withheld": bool(gated),
                      "interval_days": int(round(T)),
                      "realised_saving_pct": round(
                          100 * (base_rate - cost_rate_age(T, b_t, e_t, cp, cf)) / base_rate, 2),
                      "weight": float(fleet_n.get(cls, 1)) * base_rate})

gate = pd.DataFrame(gate_rows)
gated_fleet = float(np.average(gate.realised_saving_pct, weights=gate.weight))
ungated = float(fleet[fleet.policy == "age replacement (fitted)"].fleet_saving_pct.iloc[0])
print(f"withheld for {int(gate.withheld.sum())} of {len(gate)} classes "
      f"({', '.join(gate[gate.withheld].equipment_class)})")
print(f"ungated fleet saving {ungated:.2f}%  ->  confidence-gated {gated_fleet:.2f}%")
display(spark.createDataFrame(gate.drop(columns=["weight"])))

# Figure — realised saving by policy

plot = fleet.copy()
plot.loc[len(plot)] = ["age replacement (fitted, confidence-gated)", round(gated_fleet, 2)]
plot = plot.sort_values("fleet_saving_pct")
cols = [GREEN if "true params" in p else
        (PURPLE if "gated" in p else (MUTED if "incumbent" in p else ACCENT))
        for p in plot.policy]
fig, ax = plt.subplots(figsize=(8.4, 4.2))
ax.barh(plot.policy, plot.fleet_saving_pct, color=cols, height=.66)
ax.axvline(15, color=WARM, ls="--", lw=1.2)
ax.text(15, -0.45, " criterion 15%", color=WARM, fontsize=8)
ax.axvline(0, color=MUTED, lw=.9)
for y, v in enumerate(plot.fleet_saving_pct):
    ax.text(v + .12, y, f"{v:.1f}%", va="center", fontsize=8, color=MUTED)
ax.set_xlabel("fleet cost-rate saving vs incumbent, evaluated under true parameters (%)")
emit(fig, "a1_alternative_policies")

# Persist

from pyspark.sql.functions import lit

for name, df in [("results_alternative_policies", alt.drop(columns=["weight"])),
                 ("results_policy_fleet", fleet),
                 ("results_confidence_gate", gate.drop(columns=["weight"]))]:
    (spark.createDataFrame(df).withColumn("dataset_version", lit(DATASET_VERSION))
          .write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(tbl(name)))
    df.to_csv(f"{EXPORTS}/{name}.csv", index=False)
    print("written:", name)

print("\nFor Section 5.4: report the decomposition as ceiling vs estimation error vs policy "
      "family, using the true-parameter row as the ceiling and the fitted row as achieved.")